# Big Cats Species Classifier — Training Notebook
## SMAI Assignment 3 · T7.5 Big Cats Identifier · IIIT Hyderabad 2025-26

This notebook fine-tunes **EfficientNet-B0** on the Kaggle Big Cats Image Classification dataset.

**Dataset:** https://www.kaggle.com/datasets/patriciabrezeanu/big-cats-image-classification-dataset  
**Classes:** Cheetah, Clouded Leopard, Jaguar, Leopard, Lion, Tiger, Snow Leopard, Puma  
**Compute:** Google Colab T4 GPU (~15 minutes)  
**Output:** `big_cats_efficientnet.pth` — weights file used by `app.py`

## 1. Environment Setup

In [ ]:
# Install dependencies
!pip install timm kaggle --quiet

import os, json, shutil, random
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import timm
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 2. Download Dataset from Kaggle

Upload your `kaggle.json` API token first (Account → API → Create New Token).

In [ ]:
# Upload kaggle.json
from google.colab import files
uploaded = files.upload()  # select kaggle.json

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
# Download dataset
!kaggle datasets download -d patriciabrezeanu/big-cats-image-classification-dataset
!unzip -q big-cats-image-classification-dataset.zip -d big_cats_data

# Inspect directory structure
data_root = Path('big_cats_data')
for item in sorted(data_root.rglob('*')):
    if item.is_dir():
        count = len(list(item.glob('*.jpg'))) + len(list(item.glob('*.jpeg'))) + len(list(item.glob('*.png')))
        if count > 0:
            print(f'{item}: {count} images')

In [ ]:
# Find the actual data directory (handles nested zip structures)
def find_class_dirs(root: Path):
    """Find the directory containing class subdirectories."""
    for path in root.rglob('*'):
        if path.is_dir():
            subdirs = [d for d in path.iterdir() if d.is_dir()]
            if len(subdirs) >= 4:  # at least 4 class dirs
                imgs = list(path.glob('*/*.jpg')) + list(path.glob('*/*.jpeg')) + list(path.glob('*/*.png'))
                if len(imgs) > 100:
                    return path
    return root

DATA_DIR = find_class_dirs(data_root)
print(f'Data directory: {DATA_DIR}')
classes = sorted([d.name for d in DATA_DIR.iterdir() if d.is_dir()])
print(f'Classes ({len(classes)}): {classes}')

# Count images per class
for cls in classes:
    cls_path = DATA_DIR / cls
    imgs = list(cls_path.glob('*.jpg')) + list(cls_path.glob('*.jpeg')) + list(cls_path.glob('*.png'))
    print(f'  {cls:25s}: {len(imgs):4d} images')

## 3. Data Augmentation & Loaders

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32
VAL_SPLIT = 0.15
TEST_SPLIT = 0.10

train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
    transforms.RandomRotation(15),
    transforms.RandomGrayscale(p=0.02),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.1),
])

val_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Load full dataset
full_dataset = datasets.ImageFolder(str(DATA_DIR))
n_total = len(full_dataset)
n_val = int(n_total * VAL_SPLIT)
n_test = int(n_total * TEST_SPLIT)
n_train = n_total - n_val - n_test

# Deterministic split
torch.manual_seed(42)
train_ds, val_ds, test_ds = random_split(full_dataset, [n_train, n_val, n_test])

# Apply transforms (via wrapper)
class TransformSubset(torch.utils.data.Dataset):
    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform
    def __len__(self): return len(self.subset)
    def __getitem__(self, idx):
        img, label = self.subset[idx]
        # img is already PIL from ImageFolder
        return self.transform(img), label

train_dataset = TransformSubset(train_ds, train_transforms)
val_dataset   = TransformSubset(val_ds,   val_transforms)
test_dataset  = TransformSubset(test_ds,  val_transforms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

CLASS_NAMES = full_dataset.classes
NUM_CLASSES = len(CLASS_NAMES)
print(f'Train: {n_train} | Val: {n_val} | Test: {n_test}')
print(f'Classes: {CLASS_NAMES}')

In [ ]:
# Visualise a batch of training images
import torchvision

def imshow_batch(loader, n=8):
    imgs, labels = next(iter(loader))
    imgs = imgs[:n]
    # Denormalise
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)
    imgs = (imgs * std + mean).clamp(0, 1)
    grid = torchvision.utils.make_grid(imgs, nrow=4, padding=4)
    plt.figure(figsize=(14, 6))
    plt.imshow(grid.permute(1,2,0))
    plt.title(' | '.join(CLASS_NAMES[l] for l in labels[:n].tolist()))
    plt.axis('off')
    plt.tight_layout()
    plt.savefig('sample_batch.png', dpi=120, bbox_inches='tight')
    plt.show()

imshow_batch(train_loader)

## 4. Model — EfficientNet-B0 with Custom Head

In [ ]:
model = timm.create_model('efficientnet_b0', pretrained=True, num_classes=NUM_CLASSES)
model = model.to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters:     {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')
print(f'Model output classes: {NUM_CLASSES}')

## 5. Training with Cosine LR Schedule + Label Smoothing

In [ ]:
EPOCHS = 10
LR = 3e-4
WEIGHT_DECAY = 1e-4

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

def train_epoch(model, loader, criterion, optimizer):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * imgs.size(0)
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total += imgs.size(0)
    return running_loss / total, 100. * correct / total

def eval_epoch(model, loader, criterion):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * imgs.size(0)
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total += imgs.size(0)
    return running_loss / total, 100. * correct / total

history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_acc = 0.0

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer)
    val_loss,   val_acc   = eval_epoch(model, val_loader,   criterion)
    scheduler.step()
    
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    print(f'Epoch {epoch:02d}/{EPOCHS} | '
          f'Train Loss {train_loss:.4f} Acc {train_acc:.1f}% | '
          f'Val Loss {val_loss:.4f} Acc {val_acc:.1f}% | '
          f'LR {scheduler.get_last_lr()[0]:.2e}')
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': val_acc,
            'class_names': CLASS_NAMES,
        }, 'big_cats_efficientnet.pth')
        print(f'  ✓ Saved best model (val_acc={val_acc:.1f}%)')

print(f'\nBest validation accuracy: {best_val_acc:.1f}%')

## 6. Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history['train_loss'], label='Train', marker='o', linewidth=2)
ax1.plot(history['val_loss'],   label='Val',   marker='o', linewidth=2)
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title('Training & Validation Loss')
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(history['train_acc'], label='Train', marker='o', linewidth=2)
ax2.plot(history['val_acc'],   label='Val',   marker='o', linewidth=2)
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Training & Validation Accuracy')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.suptitle('EfficientNet-B0 — Big Cats Classifier', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: training_curves.png')

## 7. Test Set Evaluation

In [ ]:
# Load best model
checkpoint = torch.load('big_cats_efficientnet.pth', map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(device)
        outputs = model(imgs)
        _, preds = outputs.max(1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

test_acc = sum(p == l for p, l in zip(all_preds, all_labels)) / len(all_labels) * 100
print(f'Test Accuracy: {test_acc:.2f}%\n')
print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES, digits=3))

In [ ]:
# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax
)
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('True', fontsize=12)
ax.set_title('Confusion Matrix — Test Set', fontsize=14, fontweight='bold')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: confusion_matrix.png')

## 8. Ablation: Frozen Backbone vs Full Fine-Tune

In [ ]:
# Ablation A: head-only training (frozen backbone)
model_frozen = timm.create_model('efficientnet_b0', pretrained=True, num_classes=NUM_CLASSES).to(device)

# Freeze all layers except classifier
for name, param in model_frozen.named_parameters():
    if 'classifier' not in name:
        param.requires_grad = False

trainable_frozen = sum(p.numel() for p in model_frozen.parameters() if p.requires_grad)
print(f'Trainable params (frozen): {trainable_frozen:,}')

optimizer_frozen = optim.AdamW(filter(lambda p: p.requires_grad, model_frozen.parameters()), lr=1e-3)
scheduler_frozen = optim.lr_scheduler.CosineAnnealingLR(optimizer_frozen, T_max=5, eta_min=1e-6)

frozen_history = {'train_acc': [], 'val_acc': []}
for epoch in range(1, 6):
    _, tr_acc = train_epoch(model_frozen, train_loader, criterion, optimizer_frozen)
    _, vl_acc = eval_epoch(model_frozen, val_loader, criterion)
    scheduler_frozen.step()
    frozen_history['train_acc'].append(tr_acc)
    frozen_history['val_acc'].append(vl_acc)
    print(f'Epoch {epoch} | Train {tr_acc:.1f}% | Val {vl_acc:.1f}%')

print(f'\nFrozen backbone best val acc: {max(frozen_history["val_acc"]):.1f}%')
print(f'Full fine-tune best val acc:  {best_val_acc:.1f}%')

In [ ]:
# Plot ablation comparison
plt.figure(figsize=(8, 5))
plt.plot(frozen_history['val_acc'], label='Frozen backbone (head only)', marker='s', linewidth=2)
plt.plot(history['val_acc'][:5],   label='Full fine-tune (all layers)',  marker='o', linewidth=2)
plt.xlabel('Epoch'); plt.ylabel('Validation Accuracy (%)')
plt.title('Ablation: Frozen Backbone vs Full Fine-Tune')
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('ablation_frozen_vs_full.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. CLIP Zero-Shot Baseline Comparison

In [ ]:
# CLIP zero-shot evaluation for comparison
!pip install transformers --quiet
from transformers import CLIPProcessor, CLIPModel

clip_model = CLIPModel.from_pretrained('openai/clip-vit-base-patch32').to(device)
clip_processor = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')
clip_model.eval()

prompts = {
    'cheetah':        'a photo of a cheetah, the fastest land animal with solid black spots',
    'clouded_leopard': 'a photo of a clouded leopard with cloud-shaped markings in a forest',
    'jaguar':         'a photo of a jaguar, large spotted cat of the Americas with rosettes',
    'leopard':        'a photo of a leopard or Indian leopard with rosette markings',
    'lion':           'a photo of a lion or Asiatic lion, the large maned cat',
    'tiger':          'a photo of a Bengal tiger with orange coat and black stripes',
    'snow_leopard':   'a photo of a snow leopard with pale grey fur and dark rosettes in mountains',
    'puma':           'a photo of a puma, cougar or mountain lion, a tawny large cat',
}

# Pre-encode text
texts = list(prompts.values())
text_inputs = clip_processor(text=texts, return_tensors='pt', padding=True).to(device)
with torch.no_grad():
    text_features = clip_model.get_text_features(**text_inputs)
    text_features = text_features / text_features.norm(dim=-1, keepdim=True)

CLIP_CLASS_NAMES = list(prompts.keys())

# Evaluate CLIP on test set
# We need PIL images — re-load the test subset without tensor transforms
raw_dataset = datasets.ImageFolder(str(DATA_DIR))
_, _, raw_test = random_split(raw_dataset, [n_train, n_val, n_test],
                               generator=torch.Generator().manual_seed(42))

clip_correct, clip_total = 0, 0
clip_preds, clip_labels_list = [], []

for idx in range(len(raw_test)):
    img_pil, label = raw_test[idx]
    inputs = clip_processor(images=img_pil, return_tensors='pt').to(device)
    with torch.no_grad():
        img_features = clip_model.get_image_features(**inputs)
        img_features = img_features / img_features.norm(dim=-1, keepdim=True)
        scores = (img_features @ text_features.T).squeeze(0)
        pred = scores.argmax().item()
    
    # Map CLIP pred index back to dataset label
    pred_name = CLIP_CLASS_NAMES[pred]
    true_name = raw_dataset.classes[label]
    clip_preds.append(pred_name)
    clip_labels_list.append(true_name)
    if pred_name == true_name:
        clip_correct += 1
    clip_total += 1
    if idx % 50 == 0:
        print(f'  {idx}/{len(raw_test)} ...', end='\r')

clip_acc = 100. * clip_correct / clip_total
print(f'\nCLIP zero-shot test accuracy: {clip_acc:.1f}%')
print(f'EfficientNet fine-tuned accuracy: {test_acc:.1f}%')

In [ ]:
# Bar chart comparison
fig, ax = plt.subplots(figsize=(7, 4))
methods = ['CLIP\nzero-shot', 'EfficientNet-B0\nfrozen head', 'EfficientNet-B0\nfull fine-tune']
accs    = [clip_acc, max(frozen_history['val_acc']), test_acc]
colors  = ['#5C6BC0', '#FFA726', '#66BB6A']
bars = ax.bar(methods, accs, color=colors, edgecolor='white', linewidth=1.5, width=0.5)
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{acc:.1f}%', ha='center', va='bottom', fontweight='bold', fontsize=11)
ax.set_ylim(0, 105)
ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('Model Comparison on Test Set', fontsize=13, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: model_comparison.png')

## 10. Download Weights

In [ ]:
# Download all outputs
from google.colab import files
files.download('big_cats_efficientnet.pth')
files.download('training_curves.png')
files.download('confusion_matrix.png')
files.download('model_comparison.png')
files.download('ablation_frozen_vs_full.png')

print('All files downloaded!')
print(f'Place big_cats_efficientnet.pth in the same directory as app.py')

## Summary

| Metric | Value |
|--------|-------|
| Model | EfficientNet-B0 (pretrained ImageNet) |
| Epochs | 10 |
| Batch size | 32 |
| Optimizer | AdamW (lr=3e-4, wd=1e-4) |
| LR schedule | Cosine Annealing |
| Augmentation | RandomResizedCrop, HFlip, ColorJitter, RandomRotation, RandomErasing |
| Loss | CrossEntropy with label smoothing (0.1) |

Place `big_cats_efficientnet.pth` in the same directory as `app.py` before running the Streamlit app.